# Ensure ZeroBus Service Principal

Creates (or finds) a least-privilege service principal dedicated to the ZeroBus ingestion pipeline. The SPN is scoped to a single schema and secret scope — it cannot be reused for unrelated workloads.

This notebook runs as the **first task** in the `wearables_uc_setup` job. It outputs the SPN's `application_id` as a **task value** so the downstream DDL task can grant it table-level permissions.

**Secret scope keys provisioned here:**

| Key | Source | Purpose |
| --- | --- | --- |
| `client_id` | SPN `application_id` | OAuth M2M client identifier |
| `client_secret` | Generated (one-time) | OAuth M2M client secret |
| `workspace_url` | Derived from config | Databricks workspace URL for SDK/API calls |
| `zerobus_endpoint` | Derived from workspace ID + region | ZeroBus Ingest server endpoint |
| `target_table_name` | From job params | Fully qualified bronze table name |

**Permissions handled here:**
* Secret scope `READ` ACL — so the Databricks App can retrieve all secrets at runtime

**Permissions handled by the DDL task (downstream):**
* `USE CATALOG`, `USE SCHEMA`, `MODIFY`, `SELECT` on the target table

> **Credential rotation:** Delete the `client_id` and `client_secret` keys from the secret scope, then re-run this job. A new OAuth secret will be generated automatically. Non-sensitive keys (`workspace_url`, `zerobus_endpoint`, `target_table_name`) are always refreshed.

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
catalog_use = dbutils.widgets.get("catalog_use")
schema_use = dbutils.widgets.get("schema_use")
secret_scope_name = dbutils.widgets.get("secret_scope_name")

print(f"Catalog:      {catalog_use}")
print(f"Schema:       {schema_use}")
print(f"Secret scope: {secret_scope_name}")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Naming convention: dbxw-zerobus-{schema} — ties the SPN to this schema
spn_display_name = f"dbxw-zerobus-{schema_use}"
print(f"Target SPN display name: {spn_display_name}")

# Search for an existing SPN with this name
existing_spns = list(
    w.service_principals.list(filter=f'displayName eq "{spn_display_name}"')
)

if existing_spns:
    spn = existing_spns[0]
    is_new_spn = False
    print(f"Found existing SPN: {spn.display_name}")
    print(f"  application_id: {spn.application_id}")
    print(f"  id (workspace): {spn.id}")
    print(f"  active: {spn.active}")
else:
    spn = w.service_principals.create(
        display_name=spn_display_name,
        active=True
    )
    is_new_spn = True
    print(f"Created new SPN: {spn.display_name}")
    print(f"  application_id: {spn.application_id}")
    print(f"  id (workspace): {spn.id}")

spn_application_id = spn.application_id

In [0]:
# ---------------------------------------------------------------------------
# Create an OAuth secret for the SPN and store credentials in the scope.
#
# The SPN's application_id serves as the OAuth client_id.
# A new client_secret is generated ONLY when credentials don't already
# exist in the scope — this avoids unnecessary secret rotation.
#
# IMPORTANT: The client_secret is only available at creation time and
# cannot be retrieved later from the API. If lost, delete the keys
# ('client_id', 'client_secret') from the scope and re-run this job.
# ---------------------------------------------------------------------------
import re

# ---- Derive ZeroBus endpoint from workspace metadata ---------------------
# Format: https://<workspace_id>.zerobus.<region>.cloud.databricks.com
# Ref: https://docs.databricks.com/aws/en/ingestion/zerobus-ingest/

workspace_url = w.config.host.rstrip("/")
workspace_id = spark.conf.get("spark.databricks.workspaceUrl").split(".")[0] if "cloud.databricks.com" in w.config.host else None

# Get the workspace ID from the Databricks config
try:
    # On serverless / interactive clusters, the workspace ID is in the URL path
    workspace_id = w.get_workspace_id()
except Exception:
    pass

if not workspace_id:
    # Fallback: extract from the notebook context
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    workspace_id = ctx.workspaceId().get()

# Determine the region from the workspace URL
# AWS workspace URLs: https://<name>.<region>.cloud.databricks.com  or
#                     https://<name>.cloud.databricks.com (us-east-1 default)
# We need the actual AWS region for the ZeroBus endpoint.
try:
    # The cluster's Spark config has the region
    region = spark.conf.get("spark.databricks.clusterUsageTags.region", "")
except Exception:
    region = ""

if not region:
    # Fallback: try to extract from workspace URL hostname
    # Pattern: <name>.<region>.cloud.databricks.com
    host = workspace_url.replace("https://", "")
    parts = host.split(".")
    # e.g. ['fevm-hls-fde', 'cloud', 'databricks', 'com'] — no region in URL
    # e.g. ['dbc-abc123', 'us-west-2', 'cloud', 'databricks', 'com'] — has region
    if len(parts) == 5 and parts[1] not in ("cloud",):
        region = parts[1]
    else:
        region = "us-east-1"  # AWS default region

zerobus_endpoint = f"https://{workspace_id}.zerobus.{region}.cloud.databricks.com"
target_table = f"{catalog_use}.{schema_use}.wearables_zerobus"

print(f"Workspace URL:     {workspace_url}")
print(f"Workspace ID:      {workspace_id}")
print(f"Region:            {region}")
print(f"ZeroBus endpoint:  {zerobus_endpoint}")
print(f"Target table:      {target_table}")

# ---- Check if credentials already exist -----------------------------------
try:
    existing_keys = {s.key for s in w.secrets.list_secrets(scope=secret_scope_name)}
    credentials_exist = "client_id" in existing_keys and "client_secret" in existing_keys
    print(f"\nExisting secret keys in '{secret_scope_name}': {sorted(existing_keys)}")
except Exception:
    credentials_exist = False
    existing_keys = set()
    print(f"\nSecret scope '{secret_scope_name}' not accessible (may not exist yet).")

credentials_provisioned = False

if credentials_exist:
    print(f"\nOAuth credentials already provisioned \u2014 skipping rotation.")
    print("To rotate: delete 'client_id' and 'client_secret' from the scope, then re-run.")
    # Still update non-sensitive values in case they changed
    try:
        w.secrets.put_secret(scope=secret_scope_name, key="workspace_url", string_value=workspace_url)
        w.secrets.put_secret(scope=secret_scope_name, key="zerobus_endpoint", string_value=zerobus_endpoint)
        w.secrets.put_secret(scope=secret_scope_name, key="target_table_name", string_value=target_table)
        print("Updated non-sensitive secrets (workspace_url, zerobus_endpoint, target_table_name).")
    except Exception as e:
        print(f"Warning: could not update non-sensitive secrets: {e}")
    credentials_provisioned = True
else:
    print(f"\nCreating OAuth secret for SPN '{spn_display_name}' (workspace ID: {spn.id})...")
    try:
        # Generate a new OAuth client secret for the SPN
        oauth_secret = w.service_principal_secrets.create(
            service_principal_id=spn.id
        )

        # Store all secrets in the scope
        secrets_to_store = {
            "client_id":        spn.application_id,
            "client_secret":    oauth_secret.secret,
            "workspace_url":    workspace_url,
            "zerobus_endpoint": zerobus_endpoint,
            "target_table_name": target_table,
        }
        for key, value in secrets_to_store.items():
            w.secrets.put_secret(scope=secret_scope_name, key=key, string_value=value)

        print(f"Stored in scope '{secret_scope_name}':")
        print(f"  client_id         = {spn.application_id}")
        print(f"  client_secret     = {'*' * 8}... ({len(oauth_secret.secret)} chars)")
        print(f"  workspace_url     = {workspace_url}")
        print(f"  zerobus_endpoint  = {zerobus_endpoint}")
        print(f"  target_table_name = {target_table}")

        credentials_provisioned = True

    except Exception as e:
        print(f"\nError creating OAuth credentials: {e}")
        print("This may require account admin permissions.")
        print("Fallback: create credentials manually in the Account Console")
        print("and populate the secret scope via CLI:")
        print(f"  databricks secrets put-secret --scope {secret_scope_name} --key client_id")
        print(f"  databricks secrets put-secret --scope {secret_scope_name} --key client_secret")
        print(f"  databricks secrets put-secret --scope {secret_scope_name} --key zerobus_endpoint --string-value \"{zerobus_endpoint}\"")
        print(f"  databricks secrets put-secret --scope {secret_scope_name} --key workspace_url --string-value \"{workspace_url}\"")
        print(f"  databricks secrets put-secret --scope {secret_scope_name} --key target_table_name --string-value \"{target_table}\"")

In [0]:
from databricks.sdk.service.workspace import AclPermission

# Grant the SPN READ access to the secret scope so the Databricks App
# can retrieve client_id, client_secret, and endpoint credentials at runtime.
# put_acl is idempotent — safe to re-run.
try:
    w.secrets.put_acl(
        scope=secret_scope_name,
        principal=spn_application_id,
        permission=AclPermission.READ
    )
    print(f"Granted READ on scope '{secret_scope_name}' to {spn_application_id}")
except Exception as e:
    # If the scope doesn't exist yet (first deploy before bundle creates it),
    # log the error but don't fail — the scope will be created by the bundle.
    print(f"Warning: Could not set secret scope ACL: {e}")
    print("The secret scope may not exist yet. Re-run after bundle deploy.")

In [0]:
# Pass the SPN application_id to the next task in the job.
# The DDL task reads this via:
#   {{tasks.ensure_service_principal.values.spn_application_id}}
dbutils.jobs.taskValues.set(key="spn_application_id", value=spn_application_id)
print(f"Task value set: spn_application_id = {spn_application_id}")

In [0]:
print("=" * 65)
print("Service Principal Summary")
print("=" * 65)
print(f"  Display name:      {spn_display_name}")
print(f"  Application ID:    {spn_application_id}")
print(f"  Newly created:     {is_new_spn}")
print(f"  Secret scope:      {secret_scope_name}")
print(f"    READ ACL:        granted")
print(f"    Credentials:     {'provisioned' if credentials_provisioned else 'MISSING \u2014 see errors above'}")
print(f"  ZeroBus endpoint:  {zerobus_endpoint}")
print(f"  Target table:      {target_table}")
print(f"  Target schema:     {catalog_use}.{schema_use}")
print()
print("UC grants (USE CATALOG, USE SCHEMA, MODIFY, SELECT)")
print("will be applied by the downstream DDL task.")
print("=" * 65)